In [ ]:
from datetime import datetime

import gymnasium as gym
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter

from src.agents.cart_ppo import PPOCartAgent


In [2]:
LR = 2.5e-4
num_episodes = 6000
K_EPOCH = 4
REPEAT = 1

In [3]:
run_name = f"acrobot_ppo_lr{LR}_ne{num_episodes}_k{K_EPOCH}_r{REPEAT}_{datetime.now():%Y%m%d_%H%M%S}"
writer = SummaryWriter(f"./logs/{run_name}")

In [4]:
env = gym.make("Acrobot-v1", render_mode=None)
agent = PPOCartAgent(env=env, learning_rate=LR)

In [5]:
for e in range(num_episodes):
    state, _ = env.reset()

    done = False
    states = []


    # FIRST PASS:
    actions = []
    probs = []
    rewards = []
    running_batch = []
    t = 0
    while not done:
        states.append(state)

        action, prob = agent.get_action(state)

        total_reward = 0
        for _ in range(REPEAT):
            next_state, reward, terminated, truncated, _ = env.step(action)
            t += 1
            
            total_reward += reward

            done = terminated or truncated
            if done:
                break
        
        probs.append(prob)
        actions.append(action)
        rewards.append(total_reward)

        if agent.update_critic(running_batch):
            running_batch = []

        running_batch.append((state, action, reward, next_state, done))
        state = next_state



    # TRAINING PASS:
    old_log_probs = torch.stack(probs).detach()
    states_t = torch.tensor(np.array(states), dtype=torch.float32) 
    actions_t = torch.tensor(np.array(actions), dtype=torch.int8) 
        
    for _ in range(K_EPOCH):
        with torch.no_grad():
            critiques = agent.critic(states_t).squeeze(-1)
        new_log_probs = agent.evaluate_action(states_t, actions_t)
        ratio = torch.exp(new_log_probs - old_log_probs)
        agent.update(rewards, ratio, critiques)

    writer.add_scalar("duration", t, e+1)

print('Complete')

Complete


In [6]:
num_tests = 1000

test_env = gym.make("Acrobot-v1", render_mode=None)
test_env = gym.wrappers.RecordEpisodeStatistics(test_env, buffer_length=num_tests)
test_agent = PPOCartAgent(env, 0, 0)

test_agent.policy.load_state_dict(agent.policy.state_dict())


for e in range(num_tests):
    state, _ = test_env.reset()
    done = False
    total_reward = 0
    while not done:
        action = test_agent.act(state)
        next_state, reward, terminated, truncated, info = test_env.step(action)

        done = terminated or truncated
        state = next_state

    writer.add_scalar("eval/reward", info["episode"]["r"], e+1)


avg = np.average(list(test_env.return_queue))
print("avg: ", avg)

avg:  -85.498


In [7]:
writer.add_hparams(
    {"lr": LR, "num_episodes": num_episodes, "k_epoch": K_EPOCH},
    {"final_avg_reward": avg},
    run_name=".", 
)

In [8]:
writer.close()

In [ ]:
dummy_input = torch.randn(1, env.observation_space.shape[0])
torch.onnx.export(
    agent.policy,
    dummy_input,
    f"./data/{run_name}.onnx",
    input_names=["obs"],
    output_names=["action_probs"],
    dynamic_axes={"obs": {0: "batch"}, "action_probs": {0: "batch"}},
    external_data=False,
)